**Download the datasets**

In [2]:
!pip install pandas
!pip install ipython-sql prettytable

import prettytable

prettytable.DEFAULT = 'DEFAULT'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.9 MB/s eta 0:00:00


**Store the datasets in database tables**

In [3]:
import pandas as pd
import sqlite3

# Establish connection to SQLite database
conn = sqlite3.connect("FinalDB.db")

print("Connection established successfully.")

Connection established successfully.


In [4]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


**Use Pandas to load the data available in the links above to dataframes. Use these dataframes to load data on to the database FinalDB.db as required tables.**

In [8]:
import pandas as pd
import sqlite3

# Database connection
conn = sqlite3.connect("FinalDB.db")

# ✅ REAL dataset URLs
census_url = "https://data.cityofchicago.org/resource/kn9c-c2s2.csv"
schools_url = "https://data.cityofchicago.org/resource/9xs2-f89t.csv"
crime_url = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv"

# Load into DataFrames
census_df = pd.read_csv(census_url)
schools_df = pd.read_csv(schools_url)
crime_df = pd.read_csv(crime_url)

# Load into SQLite tables
census_df.to_sql("CENSUS_DATA", conn, if_exists="replace", index=False)
schools_df.to_sql("CHICAGO_PUBLIC_SCHOOLS", conn, if_exists="replace", index=False)
crime_df.to_sql("CHICAGO_CRIME_DATA", conn, if_exists="replace", index=False)

print("Data loaded successfully!")

Data loaded successfully!


**Establish a connection between SQL magic module and the database FinalDB.db**

In [6]:
%sql sqlite:///FinalDB.db

**Find the total number of crimes recorded in the CRIME table.**

In [7]:
%%sql
SELECT COUNT(*) AS TOTAL_CRIMES
FROM CHICAGO_CRIME_DATA;

 * sqlite:///FinalDB.db
(sqlite3.OperationalError) no such table: CHICAGO_CRIME_DATA
[SQL: SELECT COUNT(*) AS TOTAL_CRIMES
FROM CHICAGO_CRIME_DATA;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


**List community area names and numbers with per capita income less than 11000**

In [9]:
%%sql
SELECT community_area_name, community_area_number, per_capita_income
FROM CENSUS_DATA
WHERE per_capita_income < 11000;

 * sqlite:///FinalDB.db
(sqlite3.OperationalError) no such column: community_area_number
[SQL: SELECT community_area_name, community_area_number, per_capita_income
FROM CENSUS_DATA
WHERE per_capita_income < 11000;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


**List all case numbers for crimes involving minors?(children are not considered minors for the purposes of crime analysis)**

In [10]:
%%sql
SELECT case_number
FROM CHICAGO_CRIME_DATA
WHERE description LIKE '%MINOR%'
  AND description NOT LIKE '%CHILD%';

 * sqlite:///FinalDB.db
Done.


case_number
JK189786
JK189688
JK189599
JK189450
JK189012
JK188741
JK188681
JK188679
JK188589
JK188143


**List all kidnapping crimes involving a child?**

In [11]:
%%sql
SELECT case_number, primary_type, description
FROM CHICAGO_CRIME_DATA
WHERE primary_type = 'KIDNAPPING'
  AND UPPER(description) LIKE '%CHILD%';

 * sqlite:///FinalDB.db
Done.


case_number,primary_type,description
JK188969,KIDNAPPING,CHILD ABDUCTION / STRANGER


**List the kind of crimes that were recorded at schools. (No repetitions)**

In [12]:
%%sql
SELECT DISTINCT primary_type
FROM CHICAGO_CRIME_DATA
WHERE UPPER(location_description) LIKE '%SCHOOL%';

 * sqlite:///FinalDB.db
Done.


primary_type
WEAPONS VIOLATION
BATTERY
ASSAULT


**List the type of schools along with the average safety score for each type.**

In [13]:
%%sql
SELECT
    "Elementary, Middle, or High School" AS school_type,
    AVG(SAFETY_SCORE) AS avg_safety_score
FROM CHICAGO_PUBLIC_SCHOOLS
GROUP BY "Elementary, Middle, or High School";

 * sqlite:///FinalDB.db
Done.


school_type,avg_safety_score
"Elementary, Middle, or High School",49.50487329434698


**List 5 community areas with highest % of households below poverty line**

In [14]:
%%sql
SELECT
    community_area_name,
    percent_households_below_poverty
FROM CENSUS_DATA
ORDER BY percent_households_below_poverty DESC
LIMIT 5;

 * sqlite:///FinalDB.db
Done.


community_area_name,percent_households_below_poverty
Riverdale,56.5
Fuller Park,51.2
Englewood,46.6
North Lawndale,43.1
East Garfield Park,42.4


**Which community area is most crime prone? Display the coumminty area number only.**

In [15]:
%%sql
SELECT community_area_number
FROM CHICAGO_CRIME_DATA
GROUP BY community_area_number
ORDER BY COUNT(*) DESC
LIMIT 1;

 * sqlite:///FinalDB.db
(sqlite3.OperationalError) no such column: community_area_number
[SQL: SELECT community_area_number
FROM CHICAGO_CRIME_DATA
GROUP BY community_area_number
ORDER BY COUNT(*) DESC
LIMIT 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


**Use a sub-query to find the name of the community area with highest hardship index**

In [16]:
%%sql
SELECT community_area_name
FROM CENSUS_DATA
WHERE hardship_index = (
    SELECT MAX(hardship_index)
    FROM CENSUS_DATA
);

 * sqlite:///FinalDB.db
Done.


community_area_name
Riverdale


**Use a sub-query to determine the Community Area Name with most number of crimes?**

In [17]:
%%sql
SELECT community_area_name
FROM CENSUS_DATA
WHERE community_area_number = (
    SELECT community_area_number
    FROM CHICAGO_CRIME_DATA
    GROUP BY community_area_number
    ORDER BY COUNT(*) DESC
    LIMIT 1
);

 * sqlite:///FinalDB.db
(sqlite3.OperationalError) no such column: community_area_number
[SQL: SELECT community_area_name
FROM CENSUS_DATA
WHERE community_area_number = (
    SELECT community_area_number
    FROM CHICAGO_CRIME_DATA
    GROUP BY community_area_number
    ORDER BY COUNT(*) DESC
    LIMIT 1
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
